# Assignment–2: Machine Learning Model Development and Performance Analysis
**Course:** AD25C01 – Exploratory Data Analysis  
**Institution:** SRM TRP Engineering College, Tiruchirappalli  
**Dataset:** Team 6 cleaned online shopping dataset  
**Problem:** Predict whether a website session generates revenue (`Revenue` = 1) or not (`Revenue` = 0).

> Run this notebook in the same folder as `Team 6 - original data ,cleaned data.xlsx`, or update `DATA_PATH` below. The data already contains scaled numeric features and encoded categorical fields; this notebook preserves that representation.

In [ ]:
# If needed, install dependencies in Jupyter:
# %pip install pandas openpyxl scikit-learn matplotlib joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.inspection import permutation_importance

DATA_PATH = "Team 6 - original data ,cleaned data.xlsx"
SHEET_NAME = "Cleaned Data"
RANDOM_STATE = 42

## Part A – Problem Definition
The task is a **binary classification** problem. The target variable is `Revenue`. A value of 1 indicates a revenue-generating session and 0 indicates a session without revenue. All other columns are used as input features. The goal is to compare three classifiers and select a model based on the held-out test metrics, especially F1-score and ROC-AUC because the target classes are imbalanced.

In [ ]:
data = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
print("Dataset shape:", data.shape)
display(data.head())
print("\\nMissing values:", int(data.isna().sum().sum()))
print("Duplicate rows:", int(data.duplicated().sum()))
print("\\nTarget distribution:")
display(data["Revenue"].value_counts().rename(index={0: "No purchase (0)", 1: "Purchase (1)"}))

## Part B – Data Preparation
The supplied cleaned sheet has 9,864 rows and 27 columns. It has no missing cells or duplicate rows according to the supplied workbook. Features are already numerically encoded/scaled, so their existing representation is retained. The split is stratified to preserve the target ratio in training and test sets.

In [ ]:
TARGET = "Revenue"
X = data.drop(columns=[TARGET])
y = data[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion"))

## Part C – Model Development
Three classification models are trained: Logistic Regression, Decision Tree, and Random Forest. The imputer is included in each pipeline for reproducibility, although the supplied dataset currently has no missing values. Class weighting is used to help account for the minority purchase class.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=12, min_samples_leaf=10,
                                         class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=2,
                                         class_weight="balanced_subsample",
                                         random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

predictions = {}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = (y_pred, y_prob)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

comparison = pd.DataFrame(results).sort_values(
    ["F1-score", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(comparison.round(4))

## Part D – Model Evaluation
Accuracy measures overall correct predictions; precision measures how many predicted purchases are actual purchases; recall measures how many actual purchases are found; F1-score balances precision and recall; ROC-AUC measures ranking ability across thresholds. Confusion matrices and ROC curves help inspect errors and discrimination.

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_model = models[best_name]
best_pred, best_prob = predictions[best_name]
print("Selected model (highest F1-score, then ROC-AUC):", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred)
plt.title(f"Confusion Matrix – {best_name}")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title(f"ROC Curve – {best_name}")
plt.tight_layout()
plt.show()

## Part E – Model Comparison
Compare the models with the table above. The best model is selected using F1-score first and ROC-AUC as a tie-breaker. Accuracy alone is not sufficient because the no-purchase class is much larger than the purchase class.

In [ ]:
comparison.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]].plot(kind="bar")
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Part F – Feature Importance
Permutation importance estimates how much the chosen metric drops when one feature is shuffled. It is a model-specific diagnostic, not proof of causation. A feature may have low or negative measured importance due to correlated features, sampling variation, or the model's behaviour.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE,
    scoring="f1", n_jobs=-1
)
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)
display(importance.head(10))
top = importance.head(10).sort_values("Importance_Mean")
plt.barh(top["Feature"], top["Importance_Mean"], xerr=top["Importance_Std"])
plt.xlabel("Mean decrease in F1 after shuffling feature")
plt.title(f"Top Permutation Importance – {best_name}")
plt.tight_layout()
plt.show()

## Conclusion, Limitations and Future Improvements
- The selected model is the model with the highest test F1-score, using ROC-AUC as a tie-breaker.
- The dataset is imbalanced, so precision, recall, F1-score, and ROC-AUC should be considered alongside accuracy.
- The supplied data is already encoded and scaled; the original raw meanings/units of standardized fields cannot be reconstructed from this sheet alone.
- The dataset appears to represent website sessions, so results may not generalize to other websites or time periods.
- Future work: tune hyperparameters with cross-validation, calibrate decision thresholds, test class-resampling methods on training data only, evaluate temporal/generalization drift, and validate on a newer independent dataset.

## Save Trained Models
Uncomment the next cell to export models and the metric table to your working directory.

In [ ]:
# import os
# os.makedirs("models", exist_ok=True)
# for name, model in models.items():
#     joblib.dump(model, "models/" + name.lower().replace(" ", "_") + ".joblib")
# comparison.to_csv("model_comparison.csv", index=False)